# 04 – Scatter Plots and Correlation

Distribution analysis tells you about individual columns. But data becomes more interesting — and more useful — when you look at **two columns together**.

Questions like:
- Does studying more actually lead to better marks?
- Do students with higher attendance score higher?
- Are Math and Science scores related?

To answer these, you need to plot one variable against another, and measure how strongly they move together.

This notebook covers:
- Scatter plots — the fundamental bivariate chart
- Correlation — measuring the relationship numerically
- Correlation heatmaps — visualising all pairwise correlations at once
- Pair plots — many scatter plots in one figure

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Rebuild the dataset
rng = np.random.default_rng(42)
n = 200

df = pd.DataFrame({
    "Student_ID": range(1001, 1001 + n),
    "Gender":     rng.choice(["Male","Female"], size=n, p=[0.52, 0.48]),
    "Department": rng.choice(["Science","Arts","Commerce"], size=n, p=[0.4,0.3,0.3]),
    "City":       rng.choice(["Mumbai","Pune","Delhi","Bangalore","Chennai"], size=n),
    "Study_Hours": np.clip(rng.normal(5, 2, n).round(1), 0, 12),
    "Attendance":  np.clip(rng.normal(75, 15, n).round(1), 20, 100),
    "Math":        np.clip(rng.normal(68, 18, n).round(0).astype(int), 0, 100),
    "Science":     np.clip(rng.normal(65, 20, n).round(0).astype(int), 0, 100),
    "English":     np.clip(rng.normal(70, 15, n).round(0).astype(int), 0, 100),
    "Fees_Paid":   rng.choice([True, False], size=n, p=[0.78, 0.22]),
})
for col, frac in [("Study_Hours",0.05),("Attendance",0.03),("Math",0.04),("Science",0.04),("English",0.03)]:
    idx = rng.choice(n, size=int(n * frac), replace=False)
    df.loc[idx, col] = np.nan
df.loc[rng.choice(n, 3, replace=False), "Math"] = rng.integers(0, 15, 3)
df["Total"] = df[["Math","Science","English"]].sum(axis=1)

print("Dataset ready:", df.shape)

## 1) The Basic Scatter Plot

Each dot represents one student. The x position is their Study Hours, the y position is their Total score. If studying helps, you'd expect to see dots moving up-and-to-the-right.

In [ ]:
clean = df[["Study_Hours","Total"]].dropna()

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(clean["Study_Hours"], clean["Total"],
           color="steelblue", alpha=0.6, edgecolors="white", s=60)

ax.set_title("Study Hours vs Total Score", fontsize=14, fontweight="bold")
ax.set_xlabel("Daily Study Hours")
ax.set_ylabel("Total Score (out of 300)")
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/scatter_study_total.png", dpi=120)
plt.close()
print("Scatter plot saved.")

**How to read a scatter plot:**

- **Upward trend** (bottom-left to top-right) → positive relationship: more study = higher score
- **Downward trend** (top-left to bottom-right) → negative relationship: more of one = less of the other
- **No pattern / random cloud** → no linear relationship
- **Tight cluster** → strong relationship; **wide spread** → weak relationship

## 2) Adding a Trend Line (Regression Line)

A trend line (also called a line of best fit or regression line) shows the average relationship direction. Seaborn's `regplot()` adds it automatically.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.regplot(data=df, x="Study_Hours", y="Total",
            scatter_kws={"alpha":0.5, "color":"steelblue", "edgecolors":"white", "s":60},
            line_kws={"color":"firebrick", "linewidth":2},
            ax=ax)

ax.set_title("Study Hours vs Total Score (with Trend Line)", fontsize=13, fontweight="bold")
ax.set_xlabel("Daily Study Hours")
ax.set_ylabel("Total Score")
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/regplot_study_total.png", dpi=120)
plt.close()
print("Regression plot saved.")

`sns.regplot()` draws the scatter plot and the trend line, plus a **shaded confidence interval** around the line. The narrower the shaded band, the more confident we are about the direction.

The slope of the red line tells you the direction:
- Positive slope → studying more is associated with higher scores
- Flat line → no relationship

## 3) Scatter Plot Coloured by Category

Adding colour to encode a categorical variable turns one chart into three — you can see both the x-y relationship and how it differs by group.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colours = {"Science":"steelblue", "Arts":"darkorange", "Commerce":"seagreen"}
markers = {"Science":"o", "Arts":"s", "Commerce":"^"}

for dept in ["Science", "Arts", "Commerce"]:
    subset = df[df["Department"] == dept][["Study_Hours","Total"]].dropna()
    ax.scatter(subset["Study_Hours"], subset["Total"],
               color=colours[dept], marker=markers[dept],
               alpha=0.6, edgecolors="white", s=60, label=dept)

ax.set_title("Study Hours vs Total Score by Department", fontsize=13, fontweight="bold")
ax.set_xlabel("Daily Study Hours")
ax.set_ylabel("Total Score")
ax.legend(title="Department")
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/scatter_dept.png", dpi=120)
plt.close()
print("Coloured scatter plot saved.")

Different marker shapes (circle, square, triangle) make groups distinguishable even in black-and-white printing. Combined with colour, it's very clear which dots belong to which group.

## 4) Correlation — Measuring the Relationship

A scatter plot shows the shape of the relationship visually. **Pearson's correlation coefficient (r)** gives you a number that measures the strength and direction:

```
r = +1.0 → perfect positive linear relationship
r = +0.7 → strong positive relationship
r = +0.3 → weak positive relationship
r =  0.0 → no linear relationship
r = -0.3 → weak negative relationship
r = -0.7 → strong negative relationship
r = -1.0 → perfect negative linear relationship
```

Important caveat: correlation measures *linear* relationships only. Two variables can have a strong curved relationship and show r ≈ 0.

In [ ]:
# Compute correlation for specific pairs
clean_df = df[["Study_Hours","Attendance","Math","Science","English","Total"]].dropna()

pairs = [
    ("Study_Hours", "Total"),
    ("Attendance",  "Total"),
    ("Math",        "Science"),
    ("Math",        "English"),
]

print("Pairwise Correlations:")
print("-" * 40)
for col1, col2 in pairs:
    r = clean_df[col1].corr(clean_df[col2])
    strength = "Strong" if abs(r) > 0.6 else "Moderate" if abs(r) > 0.3 else "Weak"
    direction = "positive" if r > 0 else "negative"
    print(f"{col1:15} vs {col2:12}: r = {r:+.3f}  ({strength} {direction})")

Reading the output:
- Math and Science correlate strongly — students who are good at Math tend to be good at Science
- Study Hours has a moderate positive correlation with Total — more study hours is associated with better scores, but it's not the only factor
- Attendance may have a weaker or stronger relationship — the number will tell you

## 5) Correlation Heatmap — All Pairs at Once

When you have many numeric columns, computing each pair separately is tedious. A **correlation heatmap** shows all correlations at once, colour-coded from -1 to +1.

In [ ]:
numeric_cols = ["Study_Hours","Attendance","Math","Science","English","Total"]
corr_matrix = df[numeric_cols].corr()

print("Correlation Matrix:")
print(corr_matrix.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,            # show the number inside each cell
    fmt=".2f",             # format numbers to 2 decimal places
    cmap="coolwarm",       # blue = negative, red = positive
    center=0,              # center the colour scale at 0
    vmin=-1, vmax=1,       # fix the scale
    square=True,           # equal-sized cells
    linewidths=0.5,        # thin lines between cells
    cbar_kws={"shrink": 0.8},
    ax=ax
)

ax.set_title("Correlation Heatmap — Student Performance", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/heatmap.png", dpi=120)
plt.close()
print("Heatmap saved.")

**Reading the heatmap:**
- The diagonal is always 1.0 (a variable perfectly correlates with itself)
- The matrix is symmetric (upper-left triangle mirrors lower-right)
- **Deep red** → strong positive correlation
- **Deep blue** → strong negative correlation
- **White/pale** → near zero correlation

`cmap="coolwarm"` is a standard choice for correlation matrices. `annot=True` adds the numbers — always include them so readers can see the exact values, not just the colour.

## 6) Scatter Plot Matrix (Pair Plot)

A **pair plot** creates a grid of scatter plots for every pair of numeric columns. It's the fastest way to get an overview of all bivariate relationships at once.

In [ ]:
# Pair plot with department colour coding
plot_cols = ["Study_Hours","Attendance","Math","Science","English"]
plot_df   = df[plot_cols + ["Department"]].dropna()

palette = {"Science":"steelblue", "Arts":"darkorange", "Commerce":"seagreen"}

g = sns.pairplot(plot_df, hue="Department", palette=palette,
                 plot_kws={"alpha":0.5, "edgecolors":"none", "s":30},
                 diag_kind="kde",     # diagonal shows KDE instead of histogram
                 corner=True)         # only lower triangle (saves space)

g.figure.suptitle("Pair Plot — Student Performance", fontsize=14, fontweight="bold", y=1.02)
plt.savefig("/home/claude/05_eda/pairplot.png", dpi=100, bbox_inches="tight")
plt.close()
print("Pair plot saved.")

A pair plot is one of the first things you run when starting an EDA. In one image you can see:
- The distribution of each variable (diagonal)
- The relationship between every pair (off-diagonal)
- How groups differ in each combination (colour)

`corner=True` only plots the lower triangle — the upper triangle would be a mirror image, so this saves space without losing information.

## 7) Scatter Plots for Specific EDA Questions

Let's use scatter plots to answer concrete questions about our dataset.

In [ ]:
# Question: Is the relationship between Attendance and Math score different for
# students who paid fees vs those who didn't?

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, paid, colour, title_sfx in zip(
    axes,
    [True, False],
    ["steelblue", "firebrick"],
    ["(Fees Paid)", "(Fees Not Paid)"]
):
    subset = df[df["Fees_Paid"] == paid][["Attendance","Math"]].dropna()
    
    ax.scatter(subset["Attendance"], subset["Math"],
               color=colour, alpha=0.6, edgecolors="white", s=50)
    
    # Add trend line manually
    if len(subset) > 2:
        z = np.polyfit(subset["Attendance"], subset["Math"], 1)
        p = np.poly1d(z)
        x_line = np.linspace(subset["Attendance"].min(), subset["Attendance"].max(), 100)
        ax.plot(x_line, p(x_line), color="black", linewidth=2, linestyle="--")
    
    r = subset["Attendance"].corr(subset["Math"])
    ax.set_title(f"Attendance vs Math {title_sfx}\nr = {r:.2f}", fontweight="bold")
    ax.set_xlabel("Attendance %")
    ax.set_ylabel("Math Score")
    ax.grid(linestyle="--", alpha=0.4)

fig.suptitle("Does Fee Payment Affect the Attendance-Math Relationship?",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/05_eda/scatter_fee_comparison.png", dpi=120)
plt.close()
print("Comparison scatter plot saved.")

This kind of targeted scatter plot answers a specific business question — not just "are attendance and Math correlated?" but "does that relationship look different for different groups?"

Subplots with `sharey=True` keep the y-axis scale the same on both charts, making the comparison fair.

## 8) Correlation Doesn't Mean Causation

This is the most important caveat in data analysis.

A high correlation between Study Hours and Total Score does **not** mean:
- Studying *caused* the higher score
- Studying *more* will *always* lead to higher scores for every student

It means: **in this dataset, these two variables tend to go up and down together**.

Classic example: ice cream sales and drowning deaths are positively correlated. Both go up in summer. Hot weather *causes* both. Ice cream doesn't cause drowning.

In data analysis, always ask: *Is there a third variable (a confound) that could explain this relationship?*

In [ ]:
# Visualise a potential confound: Department
# Maybe Science students both study more AND score higher — not because of each other

dept_study = df.groupby("Department")["Study_Hours"].mean()
dept_total = df.groupby("Department")["Total"].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colours   = {"Science":"steelblue","Arts":"darkorange","Commerce":"seagreen"}

# Plot 1: Study Hours by Department
axes[0].bar(dept_study.index, dept_study.values,
            color=[colours[d] for d in dept_study.index], edgecolor="white")
axes[0].set_title("Avg Study Hours by Dept", fontweight="bold")
axes[0].set_ylabel("Hours")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# Plot 2: Total Score by Department  
axes[1].bar(dept_total.index, dept_total.values,
            color=[colours[d] for d in dept_total.index], edgecolor="white")
axes[1].set_title("Avg Total Score by Dept", fontweight="bold")
axes[1].set_ylabel("Total Score")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

# Plot 3: Scatter coloured by dept to check confounding
for dept in ["Science","Arts","Commerce"]:
    sub = df[df["Department"]==dept][["Study_Hours","Total"]].dropna()
    axes[2].scatter(sub["Study_Hours"], sub["Total"],
                    color=colours[dept], alpha=0.5, s=40, label=dept, edgecolors="none")
axes[2].set_title("Study Hours vs Total (by Dept)", fontweight="bold")
axes[2].set_xlabel("Study Hours")
axes[2].set_ylabel("Total Score")
axes[2].legend(title="Dept", fontsize=8)
axes[2].grid(linestyle="--", alpha=0.4)

fig.suptitle("Investigating Confounding Variables", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/05_eda/confound.png", dpi=120)
plt.close()
print("Confound investigation chart saved.")

If Science students study more *and* score higher — and there are more Science students in the dataset — the correlation between Study Hours and Total might be partly explained by Department, not purely by study habits.

This is exactly the kind of investigation EDA enables. You don't stop at "there's a correlation." You dig into what might explain it.

## Summary

**Scatter plots tell you:**
- Direction of a relationship (positive/negative/none)
- Strength (tight cluster vs wide spread)
- Unusual points (outliers, clusters, non-linear patterns)
- How a relationship differs across groups (via colour)

**Correlation (r) tells you:**
- -1 to +1 scale for linear relationships
- Rule of thumb: |r| < 0.3 = weak, 0.3–0.6 = moderate, > 0.6 = strong

**Heatmaps** give you the full correlation matrix at a glance.

**Pair plots** give you scatter plots for every pair of columns in one figure.

**Always remember:** Correlation ≠ Causation. Always look for confounding variables.

Next up: **Writing Insights** — how to describe and communicate what your charts are telling you.

## Practice Questions 1

1. Create a scatter plot of `Attendance` vs `Total`. Add a trend line using `sns.regplot()`. What does the correlation coefficient tell you?
2. Create a heatmap of just the three subject scores — `Math`, `Science`, `English`. Which pair has the strongest correlation?
3. Between Study Hours and Attendance, which one has a stronger correlation with Total? Use `.corr()` to find out.

## Practice Questions 2

1. Create a pair plot of `Math`, `English`, and `Study_Hours` — colour by `Gender`. Is there any visible difference between male and female students?
2. Compare the `Math vs Science` scatter plot for Science students vs non-Science students. Is there a difference in the correlation?
3. From the heatmap: identify one pair of columns with a near-zero correlation. What does that tell you about the relationship between those two variables?